In [3]:
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt

pps  = 'playground-series-s6e4/'
pds  = 'PS-s6e4-07'

train = pd.read_csv(pps + 'sample_submission.csv')

y = train['Irrigation_Need'].map({'Low':0, 'Medium':1, 'High':2}).values


def merge(list_df):
    dfs = pd.merge(list_df[0],list_df[1], on='id')
    for i in range(2,len(list_df)): 
        dfs = pd.merge(dfs,list_df[i], on='id')
    return dfs


def microEDA(x, list_names_df): 
    lwh = []
    for _p in list_names_df:
        wh = x[_p][0:1]
        if wh=='M': wh = '_'
        lwh.append(f'{wh} ')
    return ''.join(lwh) 


def vote(lns, f_vote_schema, dfs, details=True):
    dfs['wh'] = dfs.apply(lambda x: microEDA(x,lns), axis=1)
    if details:
        display(dfs.tail())
        print("'L' - Low,  '_' - Medium,  'H' - High")
        print('\n', dfs['wh'].value_counts())
    df = pd.read_csv(pps+'sample_submission.csv')
    df['Irrigation_Need'] = dfs.apply(lambda x: f_vote_schema(x,lns), axis=1)
    return df


def comparison(lns,dfs, value_target):
    dfc = dfs.drop(['id','wh'],axis=1)
    def comp(x, value_target, lns):
        return [value_target if x[ln] != value_target else None for ln in lns] 
    dfc[lns] = dfc.apply(lambda x: comp(x,value_target,lns), axis=1, result_type='expand')
    return dfc


def heatmap_show(value_target, cell_axes, color_map, dfs):
    ax = cell_axes
    ax.set_title(value_target, fontsize=10)
    sns.heatmap(dfs.isnull(), cbar=False, cmap=color_map, ax=ax)


def display_heatmaps(dfs,qp=5): # qp - q_uantity p_articipants
    if qp==4:
        equs = ['L L L L ','_ _ _ _ ','H H H H ']
    if qp==5:
        equs = ['L L L L L ','_ _ _ _ _ ','H H H H H ']
    if qp==6:
        equs = ['L L L L L L ','_ _ _ _ _ _ ','H H H H H H ']
    if qp==7:
        equs = ['L L L L L L L ','_ _ _ _ _ _ _ ','H H H H H H H ']
        
    dfs_loc = dfs.loc[~dfs['wh'].isin(equs)]
        
    dfs_low    = comparison(lns, dfs_loc, 'Low') 
    dfs_medium = comparison(lns, dfs_loc, 'Medium')
    dfs_high   = comparison(lns, dfs_loc, 'High')
    fig, axes = plt.subplots(2, 3, figsize=(11,4))
    heatmap_show('Low',    axes[1,0], "Blues",   dfs_low)
    heatmap_show('Medium', axes[1,1], "Blues",   dfs_medium)
    heatmap_show('High',   axes[1,2], "Blues",   dfs_high)
    heatmap_show('Low',    axes[0,0], "viridis", dfs_low)
    heatmap_show('Medium', axes[0,1], "viridis", dfs_medium)
    heatmap_show('High',   axes[0,2], "viridis", dfs_high)
    plt.show()


def f_vote_schema1(x, lns):       # without the last element, and the last one itself
    _lns,_1vs = lns[:-1],lns[-1] 
    if x[_lns[0]] == x[_lns[1]] == x[_lns[2]] == x[_lns[3]]: 
        return x[_lns[0]]         # if the first four are equal, hen we return them, 
    else:
        return x[_1vs]            # otherwise the last element   


def f_vote_schema2(x, lns):
    _lns,_1vs = lns[:-1],lns[-1] 
    if x[_lns[0]]==x[_lns[1]]==x[_lns[2]]==x[_lns[3]]==x[_lns[4]]==x[_lns[5]]: 
        return x[_lns[0]]
    else:
        return x[_1vs]


def f_vote_schema3(x, lns):
    _lns,_1vs = lns[:-1],lns[-1] 
    if x[_lns[0]]==x[_lns[1]]==x[_lns[2]]==x[_lns[3]]==x[_lns[4]]: 
        return x[_lns[0]]
    else:
        return x[_1vs]

In [4]:
df1 = pd.read_csv(pds+'/0.98011.csv').rename(columns={'Irrigation_Need':'df1'})
df2 = pd.read_csv(pds+'/0.98011.a.csv').rename(columns={'Irrigation_Need':'df2'})
df3 = pd.read_csv(pds+'/0.98011.b.csv').rename(columns={'Irrigation_Need':'df3'})
df4 = pd.read_csv(pds+'/0.98011.c.csv').rename(columns={'Irrigation_Need':'df4'})
df5 = pd.read_csv(pds+'/0.98026.csv').rename(columns={'Irrigation_Need':'df5'})

ldf,lns = [df1,df2,df3,df4,df5], ['df1','df2','df3','df4','df5']

weights = {
    'df1': 1.0,
    'df2': 1.0,
    'df3': 1.0,
    'df4': 1.0,
    'df5': 1.2   # best model gets more weight
}

def weighted_vote(row, cols):
    votes = {'Low':0, 'Medium':0, 'High':0}
    
    for c in cols:
        votes[row[c]] += weights[c]
    
    return max(votes, key=votes.get)

dfs   = merge(ldf)
dfv_weighted = vote(lns, weighted_vote, dfs, details=False)
dfv1 = vote (lns, f_vote_schema1, dfs, details=False); 

FileNotFoundError: [Errno 2] No such file or directory: 'PS-s6e4-07/0.98011.a.csv'

In [ ]:
#for the accuracy
df = dfv1
from sklearn.metrics import balanced_accuracy_score
y = df['Irrigation_Need'].map({'Low':0, 'Medium':1, 'High':2}).values
oof_final = dfs.copy()
oof_final['Irrigation_Need'] = dfv1['Irrigation_Need'].map({'Low':0, 'Medium':1, 'High':2}).values
accuracy = balanced_accuracy_score(y, oof_final['Irrigation_Need'])
print("Balanced Accuracy:", accuracy)
display_heatmaps(dfs,qp=5)
final_class = oof_final.argmax(axis=1)

In [ ]:
df = dfv1

In [ ]:
df.to_csv('submission_4.csv',index=False)
df